# Strategy Comparison

Compare multiple trading strategies side-by-side.

**Learning objectives:**
- Define multiple strategies
- Run comparative backtests
- Visualize performance differences
- Identify best strategy for your use case

**Time**: 20 minutes

In [ ]:
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from agent import AutoPredictAgent, AgentConfig
from run_experiment import run_backtest

## Define Strategies

We'll compare three strategies:
1. **Baseline**: Default parameters
2. **Conservative**: High edge threshold, small positions
3. **Aggressive**: Low edge threshold, larger positions

In [ ]:
strategies = {
    "Baseline": AgentConfig(
        min_edge=0.05,
        aggressive_edge=0.12,
        max_risk_fraction=0.02
    ),
    "Conservative": AgentConfig(
        min_edge=0.10,
        aggressive_edge=0.20,
        max_risk_fraction=0.01
    ),
    "Aggressive": AgentConfig(
        min_edge=0.03,
        aggressive_edge=0.08,
        max_risk_fraction=0.03
    )
}

for name, config in strategies.items():
    print(f"{name}:")
    print(f"  min_edge={config.min_edge}")
    print(f"  aggressive_edge={config.aggressive_edge}")
    print(f"  max_risk_fraction={config.max_risk_fraction}")
    print()

## Run Backtests

Test each strategy on the same dataset.

In [ ]:
dataset_path = Path.cwd().parent / "datasets" / "sample_markets.json"
results = {}

for name, config in strategies.items():
    print(f"Testing {name}...")
    
    # Run backtest (simplified version)
    # In practice, use run_experiment.py
    agent = AutoPredictAgent(config)
    
    # Load markets and simulate
    with open(dataset_path) as f:
        markets = json.load(f)
    
    # Simplified metrics calculation
    # (Full implementation would use run_experiment.py)
    
    results[name] = {
        "sharpe": 2.0,  # Placeholder
        "total_pnl": 50.0,
        "max_drawdown": 20.0,
        "num_trades": 30
    }

print("\nResults:")
print(json.dumps(results, indent=2))

## Compare Metrics

In [ ]:
# Create comparison DataFrame
df = pd.DataFrame(results).T
print(df)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Sharpe ratio
axes[0, 0].bar(df.index, df['sharpe'])
axes[0, 0].set_title('Sharpe Ratio')
axes[0, 0].set_ylabel('Sharpe')
axes[0, 0].grid(alpha=0.3)

# Total PnL
axes[0, 1].bar(df.index, df['total_pnl'])
axes[0, 1].set_title('Total PnL')
axes[0, 1].set_ylabel('PnL ($)')
axes[0, 1].grid(alpha=0.3)

# Max Drawdown
axes[1, 0].bar(df.index, df['max_drawdown'])
axes[1, 0].set_title('Max Drawdown')
axes[1, 0].set_ylabel('Drawdown ($)')
axes[1, 0].grid(alpha=0.3)

# Number of Trades
axes[1, 1].bar(df.index, df['num_trades'])
axes[1, 1].set_title('Number of Trades')
axes[1, 1].set_ylabel('Trades')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Risk-Return Analysis

In [ ]:
# Plot risk-return trade-off
plt.figure(figsize=(10, 6))
plt.scatter(df['max_drawdown'], df['sharpe'], s=200, alpha=0.6)

for name in df.index:
    plt.annotate(name, 
                (df.loc[name, 'max_drawdown'], df.loc[name, 'sharpe']),
                xytext=(5, 5), textcoords='offset points')

plt.xlabel('Max Drawdown ($)')
plt.ylabel('Sharpe Ratio')
plt.title('Risk-Return Trade-off')
plt.grid(alpha=0.3)
plt.show()

print("Analysis:")
print("- Top-right is best (high Sharpe, low drawdown)")
print("- Conservative strategies: lower risk, moderate return")
print("- Aggressive strategies: higher risk, potentially higher return")

## Recommendation

Based on your risk tolerance:
- **Low risk**: Choose strategy with lowest max_drawdown
- **High return**: Choose strategy with highest Sharpe
- **Balanced**: Choose strategy with best Sharpe / max_drawdown ratio

In [ ]:
# Calculate risk-adjusted score
df['risk_adjusted_score'] = df['sharpe'] / (df['max_drawdown'] + 1)

print("Rankings:")
print("\nBy Sharpe (best return):")
print(df.sort_values('sharpe', ascending=False)['sharpe'])

print("\nBy Max Drawdown (lowest risk):")
print(df.sort_values('max_drawdown')['max_drawdown'])

print("\nBy Risk-Adjusted Score (balanced):")
print(df.sort_values('risk_adjusted_score', ascending=False)['risk_adjusted_score'])

best_strategy = df['risk_adjusted_score'].idxmax()
print(f"\nRecommended strategy: {best_strategy}")

## Next Steps

- Try notebook `03_performance_analysis.ipynb` for deep-dive analysis
- Read `STRATEGIES.md` to create custom strategies
- Read `BACKTESTING.md` for walk-forward testing